# 12 Future Agenda and Figure Integration QA

This notebook records Subagent 12's final integration review and Fig12 drawing logic. At runtime it reads `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv`, `output/tables`, and the generated integration-check results. The current Word synchronization uses the already generated Fig12 files in four formats, so the figure does not need to be redrawn.

Outputs are restricted to:

Fig12 contract:


In [ ]:
# This cell handles basic imports and path resolution.
# Design principle: the notebook can start from the project root or code/ directory and automatically resolve back to the project root.
from pathlib import Path
from datetime import datetime
import re
import textwrap
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.pyplot as plt

# Register and require Times New Roman; stop immediately if the font is unavailable instead of using a substitute.
TIMES_NEW_ROMAN_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
]
for font_path in TIMES_NEW_ROMAN_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont("Times New Roman", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("Times New Roman is required for figure export but was not found by matplotlib.") from exc

from matplotlib.patches import Rectangle

# pandas display settings affect only notebook viewing and do not modify source files.
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# New master-table scope: N=9222, award_year covers 2010-2023 and includes 2022.
MASTER_FILENAME = "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
EXPECTED_MASTER_N = 9222
EXPECTED_AWARD_YEAR_MIN = 2010
EXPECTED_AWARD_YEAR_MAX = 2023
EXPECTED_AWARD_YEARS = list(range(EXPECTED_AWARD_YEAR_MIN, EXPECTED_AWARD_YEAR_MAX + 1))


def resolve_project_root() -> Path:
    # Identify the project root via the master data file to avoid path inconsistency when starting from code/ or the project root.
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "data" / MASTER_FILENAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate the project root: data/{MASTER_FILENAME} was not found")


ROOT = resolve_project_root()
DATA_DIR = ROOT / "data"
CODE_DIR = ROOT / "code"
OUT_DIR = ROOT / "output"
TABLE_DIR = OUT_DIR / "tables"
LOG_DIR = OUT_DIR / "logs"
FIG_DIR = OUT_DIR / "figures"

# Allowed write list for this Fig12-only revision. Integration QA/outline text remains in notebook source, but Markdown/CSV files are not written by default.
RESPONSIBLE_WRITES = [
    CODE_DIR / "12_future_agenda_and_figure_qa.ipynb",
    FIG_DIR / "Fig12_future_agenda_framework.svg",
    FIG_DIR / "Fig12_future_agenda_framework.pdf",
    FIG_DIR / "Fig12_future_agenda_framework.tiff",
    FIG_DIR / "Fig12_future_agenda_framework.png",
]
WRITE_INTEGRATION_ARTIFACTS = False

RUN_TIME = datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z")
print(f"Project root: {ROOT}")
print(f"Run time: {RUN_TIME}")


In [ ]:
# This cell only reads the master dataset, tables, and logs; it does not modify notebooks or outputs from 01-10.
# tables/logs are managed centrally in dictionaries so later QA and reports reference the same scope.
MASTER_PATH = DATA_DIR / MASTER_FILENAME
master = pd.read_csv(MASTER_PATH)

TABLE_FILES = {
    "01_quality": "01_data_quality_summary.csv",
    "02_screening": "02_screening_flow.csv",
    "03_year": "03_year_distribution.csv",
    "04_coding": "04_coding_framework.csv",
    "05_dashboard": "05_dashboard_metrics.csv",
    "06_spatial": "06_spatial_distribution.csv",
    "07_temporal_topic": "07_temporal_topic_matrix.csv",
    "08_sankey": "08_sankey_links.csv",
    "09_method_changes": "09_annual_method_changes.csv",
    "09_scale": "09_scale_heterogeneity.csv",
    "10_keyword_edges": "10_keyword_edges.csv",

    "11_wordcloud": "11_topic_wordcloud_terms.csv",
    "table1_topic": "Table1_topic_summary_matrix.csv",
}

LOG_FILES = {
    "01_data_audit": "01_data_audit.md",
    "02_screening_methods": "02_screening_methods.md",
    "03_year_trend": "03_year_trend_text.md",
    "04_coding_methods": "04_coding_methods.md",
    "05_dashboard": "05_dashboard_text.md",
    "06_spatial": "06_spatial_text.md",
    "07_temporal_topic": "07_temporal_topic_text.md",
    "08_sankey": "08_sankey_text.md",
    "09_method_scale": "09_method_scale_text.md",
    "10_topic": "10_topic_text.md",

    "11_wordcloud": "11_wordcloud_text.md",
}

# Read tables; if key tables are missing, keep the missing-file information and report it explicitly in QA.
tables = {}
missing_table_reads = []
for key, filename in TABLE_FILES.items():
    path = TABLE_DIR / filename
    if path.exists():
        tables[key] = pd.read_csv(path)
    else:
        missing_table_reads.append(str(path.relative_to(ROOT)))

# Read log text; logs supply scope notes and limitations that are not stored directly in tables.
logs = {}
missing_log_reads = []
for key, filename in LOG_FILES.items():
    path = LOG_DIR / filename
    if path.exists():
        logs[key] = path.read_text(encoding="utf-8")
    else:
        missing_log_reads.append(str(path.relative_to(ROOT)))

print(f"Master shape: {master.shape[0]} rows x {master.shape[1]} columns")
print(f"Tables loaded: {len(tables)} / {len(TABLE_FILES)}")
print(f"Logs loaded: {len(logs)} / {len(LOG_FILES)}")


In [ ]:
# This cell extracts core findings from completed 01-11 artifacts.
# These values later feed both the QA report and manuscript outline, avoiding N-value inconsistencies from manual copying.

def safe_int(value, default=None):
    # Safely convert pandas/numpy values to int; return default when missing.
    if pd.isna(value):
        return default
    try:
        return int(value)
    except Exception:
        return default


def fmt_count_pct(row, n_col="n", pct_col="percent") -> str:
    # Format n and percent values from tables consistently for direct use in Markdown reports.
    n = safe_int(row.get(n_col))
    pct = row.get(pct_col)
    if pd.isna(pct):
        return f"{row['category']} (n={n})"
    return f"{row['category']} (n={n}, {float(pct):.1f}%)"


def top_dashboard(metric_id: str, top_n: int = 3):
    # Extract the top categories for each metric from the dashboard metrics table.
    df = tables.get("05_dashboard", pd.DataFrame())
    if df.empty:
        return []
    sub = df[df["metric_id"].eq(metric_id)].sort_values(["n", "category_order"], ascending=[False, True])
    return [fmt_count_pct(row) for _, row in sub.head(top_n).iterrows()]


def extract_first_int(pattern: str, text: str, default=None):
    # Extract key N values from logs when they are not in structured tables, such as the number of records entering the Sankey analysis.
    match = re.search(pattern, text)
    if not match:
        return default
    return int(match.group(1))


master_n = len(master)
master_columns = len(master.columns)
award_year_series = pd.Series(master["award_year"]).dropna().astype(int)
years_present = sorted(award_year_series.unique().tolist())
award_year_min = min(years_present) if years_present else None
award_year_max = max(years_present) if years_present else None
award_year_range_text = f"{award_year_min}-{award_year_max}" if years_present else "not available"
missing_award_years = [year for year in EXPECTED_AWARD_YEARS if year not in years_present]
unexpected_award_years = [year for year in years_present if year not in EXPECTED_AWARD_YEARS]
award_year_range_matches = years_present == EXPECTED_AWARD_YEARS

quality = tables.get("01_quality", pd.DataFrame())
quality_fail_n = int((quality.get("status", pd.Series(dtype=str)) == "FAIL").sum()) if not quality.empty else None
quality_warn_n = int((quality.get("status", pd.Series(dtype=str)) == "WARN").sum()) if not quality.empty else None

screening = tables.get("02_screening", pd.DataFrame())
initial_n = safe_int(screening.loc[screening["stage"].eq("Identification"), "n"].iloc[0]) if not screening.empty else None
final_n = safe_int(screening.loc[screening["stage"].eq("Included"), "n"].iloc[0]) if not screening.empty else None
manual_review_n = safe_int(screening.loc[screening["stage"].eq("Manual review"), "n"].iloc[0]) if not screening.empty else None

year_dist = tables.get("03_year", pd.DataFrame())
year_sum = safe_int(year_dist["n"].sum()) if not year_dist.empty else None
peak_year_row = year_dist.sort_values("n", ascending=False).iloc[0] if not year_dist.empty else None
peak_year_text = f"{safe_int(peak_year_row['award_year'])} (n={safe_int(peak_year_row['n'])}, {float(peak_year_row['share_percent']):.2f}%)" if peak_year_row is not None else "not available"
year_dist_years = sorted(year_dist["award_year"].dropna().astype(int).unique().tolist()) if not year_dist.empty else []
year_dist_missing_award_years = [year for year in EXPECTED_AWARD_YEARS if year not in year_dist_years]
year_dist_unexpected_award_years = [year for year in year_dist_years if year not in EXPECTED_AWARD_YEARS]

dashboard_denominators = sorted(set(tables.get("05_dashboard", pd.DataFrame()).get("denominator", pd.Series(dtype=float)).dropna().astype(int).tolist()))
method_top = top_dashboard("02_method_family", 5)
object_top = top_dashboard("03_built_environment_object", 6)
performance_top = top_dashboard("04_environmental_performance", 6)
scale_place_top = top_dashboard("05_project_scale_place", 4)
application_top = top_dashboard("10_application_maturity_proxy", 4)

coding = tables.get("04_coding", pd.DataFrame())
maturity_top = []
if not coding.empty and "dimension_en" in coding.columns:
    maturity = coding[coding["dimension_en"].eq("Application maturity")].sort_values("n_records", ascending=False)
    maturity_top = [f"{row['category_en']} (n={safe_int(row['n_records'])}, {float(row['pct_records']):.1f}%)" for _, row in maturity.iterrows()]

spatial = tables.get("06_spatial", pd.DataFrame())
structured_place_empty = None
if not spatial.empty:
    coverage = spatial[spatial["section"].eq("field_coverage")].copy()
    empty_fields = {"knowledge-production place", "research-object place", "beneficiary city"}
    structured = coverage[coverage["place_name"].isin(empty_fields)]
    structured_place_empty = bool((structured["n_records"] == 0).all()) if not structured.empty else None
    city_rows = spatial[spatial["section"].eq("institution_city_distribution")].sort_values("rank").head(5)
    top_cities = [f"{row['place_name']} (n={safe_int(row['n_records'])}, {float(row['share_records'])*100:.1f}%)" for _, row in city_rows.iterrows()]
else:
    top_cities = []

temporal = tables.get("07_temporal_topic", pd.DataFrame())
topic_totals = []
if not temporal.empty:
    topic_cols = [c for c in temporal.columns if c != "award_year"]
    topic_totals_series = temporal[topic_cols].sum().sort_values(ascending=False)
    topic_totals = [f"{idx} (N={safe_int(val)})" for idx, val in topic_totals_series.head(5).items()]

sankey_log = logs.get("08_sankey", "")
sankey_entered_n = extract_first_int(r"进入路径分析的记录数：([0-9]+)", sankey_log)
sankey_excluded_n = extract_first_int(r"未进入 Sankey 的记录数：([0-9]+)", sankey_log)

method_changes = tables.get("09_method_changes", pd.DataFrame())
method_change_top = []
if not method_changes.empty:
    method_dim = method_changes[method_changes["dimension"].eq("Analytical methods")]
    top_method = method_dim.groupby("category", as_index=False)["n_records"].sum().sort_values("n_records", ascending=False).head(3)
    method_change_top = [f"{row['category']} (n={safe_int(row['n_records'])})" for _, row in top_method.iterrows()]

scale_table = tables.get("09_scale", pd.DataFrame())
scale_summary = []
if not scale_table.empty:
    scale_unique = scale_table[["scale_order", "scale", "scale_n"]].drop_duplicates().sort_values("scale_order")
    scale_summary = [f"{row['scale']} (n={safe_int(row['scale_n'])})" for _, row in scale_unique.iterrows()]

topic_matrix = tables.get("table1_topic", pd.DataFrame())
topic_cluster_sum = safe_int(topic_matrix["N"].sum()) if not topic_matrix.empty else None
top_topic_clusters = []
if not topic_matrix.empty:
    top_topic_clusters = [
        f"{row['topic_cluster']} (N={safe_int(row['N'])}, {float(row['share_percent']):.1f}%)"
        for _, row in topic_matrix.sort_values("N", ascending=False).head(3).iterrows()
    ]

keyword_edges = tables.get("10_keyword_edges", pd.DataFrame())
keyword_edge_n = len(keyword_edges) if not keyword_edges.empty else None
wordcloud_terms = tables.get("11_wordcloud", pd.DataFrame())
wordcloud_rows = len(wordcloud_terms) if not wordcloud_terms.empty else None

core_findings = {
    "master_n": master_n,
    "master_columns": master_columns,
    "years_present": years_present,
    "award_year_range_text": award_year_range_text,
    "missing_award_years": missing_award_years,
    "unexpected_award_years": unexpected_award_years,
    "award_year_range_matches": award_year_range_matches,
    "quality_fail_n": quality_fail_n,
    "quality_warn_n": quality_warn_n,
    "initial_n": initial_n,
    "final_n": final_n,
    "manual_review_n": manual_review_n,
    "year_sum": year_sum,
    "year_dist_years": year_dist_years,
    "year_dist_missing_award_years": year_dist_missing_award_years,
    "year_dist_unexpected_award_years": year_dist_unexpected_award_years,
    "peak_year_text": peak_year_text,
    "dashboard_denominators": dashboard_denominators,
    "method_top": method_top,
    "object_top": object_top,
    "performance_top": performance_top,
    "scale_place_top": scale_place_top,
    "application_top": application_top,
    "maturity_top": maturity_top,
    "structured_place_empty": structured_place_empty,
    "top_cities": top_cities,
    "topic_totals": topic_totals,
    "sankey_entered_n": sankey_entered_n,
    "sankey_excluded_n": sankey_excluded_n,
    "method_change_top": method_change_top,
    "scale_summary": scale_summary,
    "topic_cluster_sum": topic_cluster_sum,
    "top_topic_clusters": top_topic_clusters,
    "keyword_edge_n": keyword_edge_n,
}

for key, value in core_findings.items():
    print(f"{key}: {value}")


In [ ]:
# This cell draws Fig12.
# Note: all text that appears on the figure through ax.text is English; Chinese appears only in data/query strings.
# The figure uses a white background, strict grayscale, square-corner rectangles, and editable SVG/PDF text.

from matplotlib.offsetbox import AnnotationBbox, HPacker, TextArea

# Nature-style / publication-style base settings: keep SVG text as text nodes and PDF text as TrueType.
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.sans-serif": ["Times New Roman"],
    "font.monospace": ["Times New Roman"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 8,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.8,
    "legend.frameon": False,
})

FIG12_BASE = FIG_DIR / "Fig12_future_agenda_framework"

# Strict grayscale: all RGB channels are equal, avoiding color, gradients, or transparent colors.
GRAY = {
    "black": "#000000",
    "ink": "#1A1A1A",
    "text": "#303030",
    "muted": "#5C5C5C",
    "line": "#8A8A8A",
    "edge": "#4A4A4A",
    "white": "#FFFFFF",
}

SUMMARY = {
    "xy": (0.125, 0.576),
    "width": 0.750,
    "height": 0.118,
    "title": "Future research agenda",
    "subtitle": "Evidence-chain synthesis for built-environment analytics",
}

BLOCK_W = 0.375
BLOCK_H = 0.238
LEFT_X = 0.125
RIGHT_X = LEFT_X + BLOCK_W
BOTTOM_Y = 0.100
TOP_Y = BOTTOM_Y + BLOCK_H

# The four future directions come from the manuscript discussion logic; titles use full expressions rather than abbreviations.
directions = [
    {
        "number": "01",
        "title": "Validated multisource sensing",
        "bullets": [
            "Fuse satellite, street-view and POI/mobility traces",
            "Calibrate LiDAR, imagery and in situ observations",
            "Align uncertainty, bias and spatial coverage",
            "Validate proxies against built-environment constructs",
        ],
        "xy": (LEFT_X, TOP_Y),
    },
    {
        "number": "02",
        "title": "Interpretable AI linked to urban physics",
        "bullets": [
            "Tie model signals to microclimate and exposure",
            "Embed energy-balance, ventilation and causal priors",
            "Test transfer across climate and urban-form contexts",
            "Report mechanisms, limits and context validity",
        ],
        "xy": (RIGHT_X, TOP_Y),
    },
    {
        "number": "03",
        "title": "Cross-scale performance metrics",
        "bullets": [
            "Harmonize parcel, building, street and city units",
            "Track regional spillovers and boundary effects",
            "Link form metrics to carbon, heat and exposure endpoints",
            "Propagate measurement and model uncertainty",
        ],
        "xy": (LEFT_X, BOTTOM_Y),
    },
    {
        "number": "04",
        "title": "Planning and design decision support",
        "bullets": [
            "Test scenarios against actionable thresholds",
            "Quantify trade-offs, equity and risk under uncertainty",
            "Co-design workflows with planners and public-health actors",
            "Assess transferability before policy scaling",
        ],
        "xy": (RIGHT_X, BOTTOM_Y),
    },
]


def wrapped_lines(text, width):
    return textwrap.wrap(text, width=width, break_long_words=False, break_on_hyphens=False)


def bullet_block(bullets, width=56):
    lines = []
    for bullet in bullets:
        wrapped = wrapped_lines(bullet, width=width)
        if not wrapped:
            continue
        lines.append("- " + wrapped[0])
        lines.extend("  " + line for line in wrapped[1:])
    return "\n".join(lines)


def draw_rect(ax, xy, width, height, linewidth=0.95, edgecolor=None, zorder=3):
    # facecolor='none' keeps the block unfilled on the white page.
    rect = Rectangle(
        xy, width, height,
        facecolor="none",
        edgecolor=edgecolor or GRAY["edge"],
        linewidth=linewidth,
        joinstyle="miter",
        zorder=zorder,
    )
    ax.add_patch(rect)
    return rect


def add_centered_heading(ax, cx, y, number, title):
    # Pack number and title as one rendered row so the complete heading is centered.
    number_area = TextArea(number, textprops={"color": GRAY["black"], "fontsize": 7.2, "weight": "bold", "fontfamily": "Times New Roman"})
    title_area = TextArea(title, textprops={"color": GRAY["ink"], "fontsize": 8.1, "weight": "bold", "fontfamily": "Times New Roman"})
    heading = HPacker(children=[number_area, title_area], align="center", pad=0, sep=5)
    annotation = AnnotationBbox(heading, (cx, y), xycoords="data", box_alignment=(0.5, 1.0), frameon=False, pad=0, annotation_clip=False)
    annotation.set_zorder(5)
    ax.add_artist(annotation)


def draw_direction_block(ax, item, width=BLOCK_W, height=BLOCK_H):
    x, y = item["xy"]
    draw_rect(ax, (x, y), width, height)
    cx = x + width / 2
    header_y = y + height - 0.040
    add_centered_heading(ax, cx, header_y, item["number"], item["title"])
    ax.text(
        cx, y + height - 0.095,
        bullet_block(item["bullets"], width=58),
        ha="center", va="top", fontsize=7.0, linespacing=1.40,
        color=GRAY["text"], zorder=5,
    )


fig = plt.figure(figsize=(6.25, 3.60), facecolor=GRAY["white"])
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0.105, 0.895)
ax.set_ylim(BOTTOM_Y - 0.030, SUMMARY["xy"][1] + SUMMARY["height"] + 0.030)
ax.set_axis_off()

# Top summary block: use a slightly thicker border than the direction blocks below to create a clear hierarchy.
sx, sy = SUMMARY["xy"]
sw = SUMMARY["width"]
sh = SUMMARY["height"]
draw_rect(ax, (sx, sy), sw, sh, linewidth=1.20, edgecolor=GRAY["black"], zorder=4)
ax.text(sx + sw / 2, sy + sh * 0.62, SUMMARY["title"], ha="center", va="center", fontsize=11.0, weight="bold", color=GRAY["ink"], zorder=5)
ax.text(sx + sw / 2, sy + sh * 0.38, SUMMARY["subtitle"], ha="center", va="center", fontsize=8.7, color=GRAY["text"], zorder=5)

for item in directions:
    draw_direction_block(ax, item)

# Export four formats. TIFF/PNG use 600 dpi; SVG/PDF keep editable text.
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG12_BASE.with_suffix(".svg"), bbox_inches="tight", pad_inches=0.035, facecolor=GRAY["white"])
fig.savefig(FIG12_BASE.with_suffix(".pdf"), bbox_inches="tight", pad_inches=0.035, facecolor=GRAY["white"])
fig.savefig(FIG12_BASE.with_suffix(".png"), dpi=600, bbox_inches="tight", pad_inches=0.035, facecolor=GRAY["white"])
fig.savefig(FIG12_BASE.with_suffix(".tiff"), dpi=600, bbox_inches="tight", pad_inches=0.035, facecolor=GRAY["white"], pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig)

for suffix in [".svg", ".pdf", ".tiff", ".png"]:
    path = FIG12_BASE.with_suffix(suffix)
    print(f"{path.relative_to(ROOT)}: {path.stat().st_size} bytes")


In [ ]:
# This cell runs QA checks on figures, notebooks, key tables, and logs; this Fig12-only revision does not write figure_inventory.csv by default.
# Automatic checks cover expected-file existence, nonzero figure files, and obvious Chinese characters in visible SVG text.

EXPECTED_NOTEBOOKS = [
    "01_data_loading_and_quality_audit.ipynb",
    "02_screening_workflow_and_sample_construction.ipynb",
    "03_annual_trend_analysis.ipynb",
    "04_coding_system_and_conceptual_framework.ipynb",
    "05_review_overview_dashboard.ipynb",
    "06_spatial_distribution_analysis.ipynb",
    "07_topic_yearly_evolution_heatmap.ipynb",
    "08_data_method_performance_pathways.ipynb",
    "09_technology_change_and_scale_heterogeneity.ipynb",
    "10_topic_clustering_and_keyword_network.ipynb",

    "11_topic_word_cloud_analysis.ipynb",
    "12_future_agenda_and_figure_qa.ipynb",
]

# EXPECTED_FIGURES tracks source export filenames; manuscript captions use MAIN_TEXT_FIGURE_ORDER by document-flow order.
EXPECTED_FIGURES = [
    ("Fig1", "Fig1_annual_growth"),
    ("Fig2", "Fig2_screening_workflow"),
    ("Fig3", "Fig3_conceptual_framework"),
    ("Fig4", "Fig4_overview_dashboard"),
    ("Fig5", "Fig5_spatial_distribution"),
    ("Fig6", "Fig6_temporal_topic_heatmap"),
    ("Fig7", "Fig7_data_method_performance_sankey"),
    ("Fig8", "Fig8_annual_method_changes"),
    ("Fig9", "Fig9_scale_heterogeneity"),
    ("Fig10", "Fig10_topic_network"),

    ("Fig11", "Fig11_thematic_wordclouds"),
    ("Fig12", "Fig12_future_agenda_framework"),
]

MAIN_TEXT_FIGURE_ORDER = [
    ("Figure 1", "Fig2_screening_workflow", "screening workflow"),
    ("Figure 2", "Fig3_conceptual_framework", "conceptual framework"),
    ("Figure 3", "Fig1_annual_growth", "annual growth"),
    ("Figure 4", "Fig4_overview_dashboard", "overview dashboard"),
    ("Figure 5", "Fig5_spatial_distribution", "spatial distribution"),
    ("Figure 6", "Fig6_temporal_topic_heatmap", "temporal topic heatmap"),
    ("Figure 7", "Fig7_data_method_performance_sankey", "data-method-performance Sankey"),
    ("Figure 8", "Fig8_annual_method_changes", "annual method changes"),
    ("Figure 9", "Fig9_scale_heterogeneity", "scale heterogeneity"),
    ("Figure 10", "Fig10_topic_network", "topic network"),
    ("Figure 11", "Fig11_thematic_wordclouds", "thematic word clouds"),
    ("Figure 12", "Fig12_future_agenda_framework", "future agenda framework"),
]
EXPECTED_FORMATS = ["svg", "pdf", "tiff", "png"]

EXPECTED_TABLES = [
    "01_data_quality_summary.csv",
    "02_screening_flow.csv",
    "03_year_distribution.csv",
    "04_coding_framework.csv",
    "05_dashboard_metrics.csv",
    "06_spatial_distribution.csv",
    "07_temporal_topic_matrix.csv",
    "08_sankey_links.csv",
    "09_annual_method_changes.csv",
    "09_scale_heterogeneity.csv",
    "10_keyword_edges.csv",

    "11_topic_wordcloud_terms.csv",
    "Table1_topic_summary_matrix.csv",
]

EXPECTED_LOGS = [
    "01_data_audit.md",
    "02_screening_methods.md",
    "03_year_trend_text.md",
    "04_coding_methods.md",
    "05_dashboard_text.md",
    "06_spatial_text.md",
    "07_temporal_topic_text.md",
    "08_sankey_text.md",
    "09_method_scale_text.md",
    "10_topic_text.md",

    "11_wordcloud_text.md",
]

# Check only the CJK Unified Ideographs range to avoid treating English punctuation, math symbols, or metadata as Chinese.
CJK_RE = re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF]")
VISIBLE_SVG_TAGS = {"text", "tspan", "title", "desc"}
SVG_FONT_RE = re.compile(r"(Times New Roman|serif)", re.IGNORECASE)


def inspect_svg_visible_cjk(svg_path: Path) -> dict:
    # Parse visible/semantic SVG text nodes; do not scan path coordinates, style, or metadata.
    result = {"parse_error": "", "text_node_count": 0, "cjk_hits": []}
    try:
        root = ET.parse(svg_path).getroot()
    except Exception as exc:
        result["parse_error"] = str(exc)
        return result

    for elem in root.iter():
        tag = elem.tag.split("}", 1)[-1]
        if tag not in VISIBLE_SVG_TAGS:
            continue
        for text_part in [elem.text, elem.tail]:
            if not text_part:
                continue
            cleaned = " ".join(text_part.split())
            if not cleaned:
                continue
            result["text_node_count"] += 1
            if CJK_RE.search(cleaned):
                result["cjk_hits"].append(cleaned[:80])
    return result


def inspect_svg_text_font(svg_path: Path) -> dict:
    # Check whether SVG text nodes explicitly declare Times New Roman or a serif font family.
    result = {"parse_error": "", "text_tag_count": 0, "font_hit_count": 0}
    try:
        root = ET.parse(svg_path).getroot()
    except Exception as exc:
        result["parse_error"] = str(exc)
        return result

    for elem in root.iter():
        tag_name = elem.tag.split("}")[-1]
        if tag_name != "text":
            continue
        result["text_tag_count"] += 1
        attr_text = " ".join(str(value) for value in elem.attrib.values())
        if SVG_FONT_RE.search(attr_text):
            result["font_hit_count"] += 1
    return result


missing_notebooks = [name for name in EXPECTED_NOTEBOOKS if not (CODE_DIR / name).exists()]
missing_tables = [name for name in EXPECTED_TABLES if not (TABLE_DIR / name).exists()]
missing_logs = [name for name in EXPECTED_LOGS if not (LOG_DIR / name).exists()]

inventory_rows = []
figure_missing_items = []
figure_zero_items = []
svg_cjk_issues = []
svg_parse_issues = []
svg_textless_items = []
svg_font_issues = []
svg_font_parse_issues = []

for fig_id, stem in EXPECTED_FIGURES:
    existing_formats = []
    missing_formats = []
    size_parts = []
    issue_notes = []
    svg_cjk_note = "SVG visible CJK: not checked"
    svg_font_note = "SVG font: not checked"

    for fmt in EXPECTED_FORMATS:
        path = FIG_DIR / f"{stem}.{fmt}"
        if path.exists():
            size = path.stat().st_size
            existing_formats.append(fmt)
            size_parts.append(f"{fmt}={size}")
            if size <= 0:
                figure_zero_items.append(f"{fig_id}.{fmt}")
                issue_notes.append(f"{fmt} zero bytes")
        else:
            missing_formats.append(fmt)
            figure_missing_items.append(f"{fig_id}.{fmt}")

    svg_path = FIG_DIR / f"{stem}.svg"
    if svg_path.exists() and svg_path.stat().st_size > 0:
        svg_result = inspect_svg_visible_cjk(svg_path)
        if svg_result["parse_error"]:
            svg_parse_issues.append(f"{fig_id}: {svg_result['parse_error']}")
            issue_notes.append("SVG parse error for visible-text check")
        else:
            if svg_result["text_node_count"] == 0:
                svg_textless_items.append(fig_id)
                issue_notes.append("SVG text nodes not detected; editable text cannot be verified automatically")
            if svg_result["cjk_hits"]:
                hit_text = "; ".join(svg_result["cjk_hits"][:3])
                svg_cjk_issues.append(f"{fig_id}: {hit_text}")
                issue_notes.append(f"SVG visible CJK detected: {hit_text}")
            else:
                svg_cjk_note = "SVG visible CJK: none"

        font_result = inspect_svg_text_font(svg_path)
        if font_result["parse_error"]:
            svg_font_parse_issues.append(f"{fig_id}: {font_result['parse_error']}")
            issue_notes.append("SVG parse error for font check")
        elif font_result["text_tag_count"] == 0:
            svg_font_issues.append(f"{fig_id}: no SVG text tags for font verification")
            issue_notes.append("SVG font not verified: no text tags")
        elif font_result["font_hit_count"] != font_result["text_tag_count"]:
            svg_font_issues.append(
                f"{fig_id}: {font_result['font_hit_count']}/{font_result['text_tag_count']} text tags declare Times New Roman or serif"
            )
            issue_notes.append("SVG font declaration incomplete")
        else:
            svg_font_note = "SVG font: Times New Roman/serif"

    note_parts = issue_notes + [svg_cjk_note, svg_font_note]
    notes = "; ".join(note_parts) if issue_notes else f"OK; {svg_cjk_note}; {svg_font_note}"
    inventory_rows.append({
        "figure_id": fig_id,
        "expected_formats": ";".join(EXPECTED_FORMATS),
        "existing_formats": ";".join(existing_formats),
        "missing_formats": ";".join(missing_formats),
        "file_sizes": "; ".join(size_parts),
        "notes": notes,
    })

inventory_df = pd.DataFrame(inventory_rows, columns=["figure_id", "expected_formats", "existing_formats", "missing_formats", "file_sizes", "notes"])
inventory_path = OUT_DIR / "figure_inventory.csv"
if WRITE_INTEGRATION_ARTIFACTS:
    inventory_df.to_csv(inventory_path, index=False, encoding="utf-8-sig")

# N-value consistency checks: only verify that module-declared denominators match their corresponding analytical scopes.
n_checks = [
    {"check": "master_record_count", "status": "PASS" if master_n == EXPECTED_MASTER_N else "FAIL", "detail": f"master n={master_n}; expected n={EXPECTED_MASTER_N}"},
    {"check": "master_award_year_range", "status": "PASS" if award_year_range_matches else "FAIL", "detail": f"award years={award_year_range_text}; missing={missing_award_years}; unexpected={unexpected_award_years}"},
    {"check": "master_column_count", "status": "PASS" if master_columns > 0 else "FAIL", "detail": f"columns={master_columns}"},
    {"check": "screening_final_matches_master", "status": "PASS" if final_n == master_n else "FAIL", "detail": f"screening final n={final_n}; master n={master_n}"},
    {"check": "year_distribution_sums_to_master", "status": "PASS" if year_sum == master_n else "FAIL", "detail": f"year table sum={year_sum}; master n={master_n}"},
    {"check": "year_distribution_award_year_range", "status": "PASS" if year_dist_years == EXPECTED_AWARD_YEARS else "FAIL", "detail": f"year table years={year_dist_years[:1]}..{year_dist_years[-1:]}; missing={year_dist_missing_award_years}; unexpected={year_dist_unexpected_award_years}"},
    {"check": "dashboard_denominator", "status": "PASS" if dashboard_denominators == [master_n] else "FAIL", "detail": f"dashboard denominators={dashboard_denominators}"},
    {"check": "topic_clusters_sum_to_master", "status": "PASS" if topic_cluster_sum == master_n else "FAIL", "detail": f"topic cluster sum={topic_cluster_sum}; master n={master_n}"},
    {"check": "sankey_documented_subset", "status": "PASS" if sankey_entered_n and sankey_excluded_n and sankey_entered_n + sankey_excluded_n == master_n else "WARN", "detail": f"Sankey entered n={sankey_entered_n}; excluded n={sankey_excluded_n}; master n={master_n}"},
    {"check": "quality_no_fail", "status": "PASS" if quality_fail_n == 0 else "FAIL", "detail": f"quality FAIL={quality_fail_n}; WARN={quality_warn_n}"},
]
n_checks_df = pd.DataFrame(n_checks)

hard_failures = []
if missing_notebooks:
    hard_failures.append("missing notebooks")
if figure_missing_items:
    hard_failures.append("missing figure formats")
if figure_zero_items:
    hard_failures.append("zero-size figure files")
if svg_cjk_issues:
    hard_failures.append("visible CJK in SVG")
if svg_font_issues or svg_font_parse_issues:
    hard_failures.append("SVG font not verified as Times New Roman/serif")
if missing_tables:
    hard_failures.append("missing key tables")
if missing_logs:
    hard_failures.append("missing integration notes")
if (n_checks_df["status"] == "FAIL").any():
    hard_failures.append("N consistency failure")

qa_hard_pass = len(hard_failures) == 0
qa_status = "PASS_WITH_WARNINGS" if qa_hard_pass else "FAIL"

if WRITE_INTEGRATION_ARTIFACTS:
    print(f"Inventory written: {inventory_path.relative_to(ROOT)}")
else:
    print("Figure inventory kept in memory; CSV write skipped for Fig12-only revision.")
print(f"QA hard pass: {qa_hard_pass}")
print(inventory_df.to_string(index=False))
print(n_checks_df.to_string(index=False))


In [ ]:
# This cell prepares integration-report text; this Fig12-only revision does not write Markdown files by default.
# The report covers completion status, missing items, N-value consistency, year coverage, English-label checks, and main limitations.

def bullet_list(items, empty_text="None"):
    # Markdown list helper: write an explicit None for empty lists so report readers do not have to guess.
    if not items:
        return f"- {empty_text}"
    return "\n".join(f"- {item}" for item in items)


def join_items(items, empty_text="not available"):
    # Compress several core findings into one sentence for the integration report and outline.
    return "; ".join(items) if items else empty_text


missing_items = []
missing_items += [f"Notebook missing: code/{name}" for name in missing_notebooks]
missing_items += [f"Figure format missing: {item}" for item in figure_missing_items]
missing_items += [f"Key table missing: output/tables/{name}" for name in missing_tables]
missing_items += [f"Integration note missing: {name}" for name in missing_logs]
missing_items += [f"SVG font issue: {item}" for item in svg_font_issues]
missing_items += [f"SVG font parse issue: {item}" for item in svg_font_parse_issues]

english_label_lines = []
if svg_cjk_issues:
    english_label_lines.append("Visible CJK characters were detected in SVG text nodes:")
    english_label_lines.extend([f"  - {item}" for item in svg_cjk_issues])
else:
    english_label_lines.append("No obvious visible Chinese characters were detected in SVG text/title/desc nodes for Fig1-Fig12.")
if svg_textless_items:
    english_label_lines.append("Some SVG files have no detected text nodes, so editable text cannot be automatically verified: " + ", ".join(svg_textless_items) + ".")
if svg_parse_issues:
    english_label_lines.append("Some SVG files could not be parsed for visible-text checking: " + "; ".join(svg_parse_issues) + ".")
english_label_lines.append("This check does not judge whether every English label is semantically correct; it only detects obvious visible CJK characters in SVG text-like nodes.")

font_check_lines = []
if svg_font_issues or svg_font_parse_issues:
    font_check_lines.append("SVG font verification issues were detected:")
    font_check_lines.extend([f"  - {item}" for item in svg_font_issues])
    font_check_lines.extend([f"  - {item}" for item in svg_font_parse_issues])
else:
    font_check_lines.append("All detected SVG text tags for Fig1-Fig12 declare Times New Roman or serif.")
font_check_lines.append("This check inspects SVG text tag style/font attributes; it does not visually audit PDF font embedding.")

unresolved_items = [
    "The synthesis connects the curated record set, award-year coverage, coding dimensions and thematic modules, but it cannot independently prove upstream retrieval completeness outside the curated dataset.",
    "Keyword and rule-based category assignments may include broad-term false positives; final topical inclusion still needs domain review for boundary cases.",
    "Structured study-place, knowledge-production-place and beneficiary-city fields are empty, so spatial findings describe institution geography rather than verified study areas.",
    "Figure labels and graphical design should still be reviewed for semantic accuracy, readability, accessibility and editorial fit.",
    "Sankey, annual method-change and scale analyses use documented derived denominators or single-label rules, so they should not be merged without preserving each module's denominator.",
]

inventory_status_line = (
    f"- The manuscript figure sequence summarizes {len(inventory_df)} evidence-chain figures."
    if WRITE_INTEGRATION_ARTIFACTS
    else f"- The manuscript figure sequence summarizes {len(inventory_df)} evidence-chain figures."
)
outline_status_line = (
    f"- The manuscript outline is aligned with the evidence-chain narrative."
    if WRITE_INTEGRATION_ARTIFACTS
    else "- The manuscript outline is aligned with the evidence-chain narrative."
)

qa_status_display = "passed with warnings" if qa_status == "PASS_WITH_WARNINGS" else "failed"

qa_lines = [
    "# Integration report for final assembly", "",
    f"Generated: {RUN_TIME}", "",
    "## Completion status", "",
    f"- Overall integration status: **{qa_status_display}**.",
    f"- Master dataset: `{MASTER_PATH.relative_to(ROOT)}` with {master_n} deduplicated records and {master_columns} columns; expected N={EXPECTED_MASTER_N}.",
    f"- Master award-year coverage: {award_year_range_text}; expected {EXPECTED_AWARD_YEAR_MIN}-{EXPECTED_AWARD_YEAR_MAX}; missing years: {missing_award_years or 'none'}; unexpected years: {unexpected_award_years or 'none'}.",
    "- Figure 12 closes the synthesis by linking annual trends, coding dimensions, thematic clusters and future research priorities.",
    inventory_status_line,
    outline_status_line,
    "- The synthesis narrative should remain centered on the evidence base, coding framework, thematic findings and future agenda.", "",
    "## N-value consistency", "",
]

for _, row in n_checks_df.iterrows():
    check_label = str(row["check"]).replace("_", " ").replace("award year", "award-year")
    qa_lines.append(f"- **{row['status']}** {check_label}: {row['detail']}.")

qa_lines += [
    "", "## Core findings carried forward", "",
    f"- Screening: initial NSFC records N={initial_n}; final included N={final_n}; priority manual-review subset N={manual_review_n}.",
    f"- Annual trend: master award-year coverage is {award_year_range_text}; records peak in {peak_year_text}; the year table sums to N={year_sum}.",
    f"- Dashboard method families: {join_items(method_top)}.",
    f"- Built-environment objects: {join_items(object_top)}.",
    f"- Environmental performance labels: {join_items(performance_top)}.",
    f"- Application/maturity proxy: {join_items(application_top)}.",
    f"- Spatial concentration: top institution cities include {join_items(top_cities)}.",
    f"- Temporal topic totals: {join_items(topic_totals)}.",
    f"- Sankey subset: N={sankey_entered_n} entered data-method-performance path analysis; N={sankey_excluded_n} lacked explicit performance evidence for this module.",
    f"- Method-change signal: leading analytical methods include {join_items(method_change_top)}.",
    f"- Scale heterogeneity: {join_items(scale_summary)}.",
    f"- Topic clusters: {join_items(top_topic_clusters)}; topic-cluster N sums to {topic_cluster_sum}.",
    f"- Keyword network: {keyword_edge_n} complete weighted co-occurrence edges were generated before Fig10 readability filtering.", "",
    "## Figure language review", "",
]
qa_lines.extend(english_label_lines)
qa_lines += ["", "## Figure typography review", ""]
qa_lines.extend(font_check_lines)
qa_lines += [
    "", "## Master dataset coverage", "",
    f"- Curated dataset path: `{MASTER_PATH.relative_to(ROOT)}`.",
    f"- Expected master scope: N={EXPECTED_MASTER_N}; award years {EXPECTED_AWARD_YEAR_MIN}-{EXPECTED_AWARD_YEAR_MAX}.",
    f"- Observed master scope: N={master_n}; award years {award_year_range_text}; missing years: {missing_award_years or 'none'}; unexpected years: {unexpected_award_years or 'none'}.",
    f"- Downstream annual table years: {year_dist_years}; missing expected years: {year_dist_missing_award_years or 'none'}; unexpected years: {year_dist_unexpected_award_years or 'none'}.", "",
    "## Main limitations", "",
    bullet_list(unresolved_items), "",
    "## Notebook-managed manuscript artifacts", "",
]
qa_lines.extend([f"- `{path.relative_to(ROOT)}`" for path in RESPONSIBLE_WRITES if path.name != "12_future_agenda_and_figure_qa.ipynb"])

qa_report_path = OUT_DIR / "qa_report.md"
if WRITE_INTEGRATION_ARTIFACTS:
    qa_report_path.write_text("\n".join(qa_lines) + "\n", encoding="utf-8")
    print(f"Integration report written: {qa_report_path.relative_to(ROOT)}")
else:
    print("Integration report Markdown write skipped for Fig12-only revision.")


In [ ]:
# This cell prepares manuscript_outline.md text; this Fig12-only revision does not write Markdown files by default.
# The outline is organized by Introduction / Methods / Results / Discussion / Conclusion and embeds the placement and finding sentence for each figure and key table.

outline_lines = [
    "# Manuscript outline", "",
    f"Generated: {RUN_TIME}", "",
    "## Introduction", "",
    "- Establish the review problem: built-environment analytics increasingly connects remote sensing, urban sensing traces, AI and performance-oriented urban planning, but the evidence remains fragmented across data sources, scales and application maturity.",
    "- Lock the evidence-layer scope early: this is a review-style synthesis that uses NSFC final-database project records to map built-environment analytics progress and trends, not a study of the funding database itself or a manually verified article-level systematic review.",
    "- Use **main-text Figure 2 Conceptual framework** after the problem statement. Key finding sentence: Built-environment analytics evidence can be organized as a chain from sensing data sources to analytical methods, built-environment objects, environmental performance and application maturity.",
    "- Use **output/tables/04_coding_framework.csv** as a Methods-facing coding reference. Key finding sentence: The coding framework contains five dimensions covering built-environment objects, sensing data sources, analytical methods, environmental performance and application maturity.", "",
    "## Methods", "",
    f"- Use **main-text Figure 1 Screening workflow** to document sample construction. Key finding sentence: Records were retrieved through the NSFC final-project advanced search/data retrieval interface, deduplicated at project level and consolidated into N={final_n} project records relevant to built-environment analytics, with N={manual_review_n} priority manual-review records retained as a boundary-review subset rather than excluded.",
    "- Use **output/tables/02_screening_flow.csv** for the screening sequence. Key finding sentence: The positive rule is method terms AND built-environment/object or performance terms, followed by exclusion filters and project-level deduplication.",
    f"- Use **output/tables/01_data_quality_summary.csv** for Methods data-quality reporting. Key finding sentence: The final analytical dataset should pass N={EXPECTED_MASTER_N} plus award-year {EXPECTED_AWARD_YEAR_MIN}-{EXPECTED_AWARD_YEAR_MAX} coverage checks after downstream regeneration.",
    "- Use **output/tables/06_spatial_distribution.csv** to explain spatial inference limits. Key finding sentence: Structured place information is empty, so spatial analysis uses inferred institution city and province derived from recipient-organization text.", "",
    "## Results", "",
    f"- Use **main-text Figure 3 Annual growth** at the start of Results. Key finding sentence: Annual records cover the award-year window {award_year_range_text} and peak in {peak_year_text}.",
    f"- Use **output/tables/03_year_distribution.csv** with main-text Figure 3. Key finding sentence: The year-distribution table should sum to N={master_n} and cover each award year from {EXPECTED_AWARD_YEAR_MIN} to {EXPECTED_AWARD_YEAR_MAX}.",
    f"- Use **main-text Figure 4 Overview dashboard** for evidence-layer composition. Key finding sentence: The method-family signal is led by {join_items(method_top[:3])}, with environmental performance absent or unreported for a substantial subset.",
    f"- Use **output/tables/05_dashboard_metrics.csv** with main-text Figure 4. Key finding sentence: Dashboard denominators are consistently N={master_n}, enabling direct comparison of single-label method, object, performance, scale and maturity summaries.",
    f"- Use **main-text Figure 5 Spatial distribution** for institution geography. Key finding sentence: Institution locations concentrate in {join_items(top_cities[:5])}, but these are not verified study-area or beneficiary-city locations.",
    f"- Use **main-text Figure 6 Temporal topic heatmap** for topic evolution. Key finding sentence: The strongest cumulative topic signals are {join_items(topic_totals[:3])} across the {EXPECTED_AWARD_YEAR_MIN}-{EXPECTED_AWARD_YEAR_MAX} award-year window.",
    "- Use **output/tables/07_temporal_topic_matrix.csv** with main-text Figure 6. Key finding sentence: Topic counts are record-level matches and are non-exclusive across topic columns.",
    f"- Use **main-text Figure 7 Data-method-performance Sankey** for evidence pathways. Key finding sentence: N={sankey_entered_n} records enter the path analysis, linking geospatial or remote-sensing data streams to simulation, retrieval, fusion, AI and performance endpoints.",
    "- Use **output/tables/08_sankey_links.csv** with main-text Figure 7. Key finding sentence: Fractional weighting preserves one unit of contribution per record within each path segment, preventing multilabel records from dominating the flow.",
    f"- Use **main-text Figure 8 Annual method changes** for technical shifts. Key finding sentence: Leading analytical methods include {join_items(method_change_top)}, with later-year changes interpreted across the full {EXPECTED_AWARD_YEAR_MIN}-{EXPECTED_AWARD_YEAR_MAX} award-year window.",
    "- Use **output/tables/09_annual_method_changes.csv** with main-text Figure 8. Key finding sentence: Each record is compressed to one primary category per dimension to keep annual 100% bars interpretable.",
    f"- Use **main-text Figure 9 Scale heterogeneity** for scale-specific patterns. Key finding sentence: Scale distribution spans {join_items(scale_summary)}, and each scale shows different data-source and performance-target mixes.",
    "- Use **output/tables/09_scale_heterogeneity.csv** with main-text Figure 9. Key finding sentence: The scale analysis uses text-derived primary scale labels rather than verified geocoded study places.",
    f"- Use **main-text Figure 10 Topic network** for synthesis of thematic structure. Key finding sentence: Six topic clusters sum to N={topic_cluster_sum}, led by {join_items(top_topic_clusters[:2])}.",
    f"- Use **output/tables/Table1_topic_summary_matrix.csv** as Table 1 in the manuscript. Key finding sentence: Table 1 translates each topic cluster into data sources, methods, performance indicators, scales, validation/application proxies and representative keywords.",
    f"- Use **output/tables/10_keyword_edges.csv** with main-text Figure 10. Key finding sentence: The full keyword network contains {keyword_edge_n} weighted co-occurrence edges before readability filtering in the rendered figure.",
    "- Use **main-text Figure 11 Thematic word clouds** to reveal cluster-specific lexical signatures. Key finding sentence: The six thematic streams are distinguished by controlled English terms derived from visible project metadata.",
    f"- Use **output/tables/11_topic_wordcloud_terms.csv** with main-text Figure 11. Key finding sentence: Word size is based on cluster-specific log-odds weighted by term frequency and cross-cluster specificity across {wordcloud_rows} topic-term rows, not simple corpus-wide frequency.", "",
    "## Discussion", "",
    "- Use **main-text Figure 12 Future agenda framework** as the closing synthesis figure. Key finding sentence: Figure 12 synthesizes the future agenda as an evidence-chain extension rather than a separate classification, linking the review evidence to validated multisource sensing, interpretable AI linked to urban physics, cross-scale performance metrics and planning and design decision support.",
    "- Use the Figure 12 top synthesis block and four lower agenda blocks explicitly: satellite, street-view, POI and mobility traces calibrated with LiDAR and in situ observations; microclimate and exposure models constrained by energy-balance, ventilation and causal priors; parcel, building, street and city metrics linked to carbon, heat and exposure endpoints; and decision support built around actionable thresholds, trade-offs, equity, risk and transferability before policy scaling.",
    "- Use the Figure 12 caption logic consistently: Future research agenda derived from the evidence-chain synthesis; the top block extends the review evidence into the four next-step priorities.",
    "- Discuss translational implications: uncertainty, bias, spatial coverage, boundary effects, measurement/model uncertainty, practitioner co-design and context validity should be reported as first-order outputs before planning or policy scaling.", "",
    "## Conclusion", "",
    f"- Conclude that the review uses N={master_n} deduplicated NSFC project records as an evidence layer to build a reproducible map of built-environment analytics progress and trends, covering annual growth, screening, conceptual structure, evidence-layer composition, institution geography, temporal topics, data-method-performance pathways, technical change, scale heterogeneity, keyword clusters, thematic word clouds and future agenda.",
    "- State the main limitation plainly: the synthesis connects annual trends, coding dimensions, thematic clusters and the future agenda, but it cannot replace domain review of classification boundaries, upstream retrieval completeness or study-place validity.",
]

outline_path = OUT_DIR / "manuscript_outline.md"
if WRITE_INTEGRATION_ARTIFACTS:
    outline_path.write_text("\n".join(outline_lines) + "\n", encoding="utf-8")
    print(f"Manuscript outline written: {outline_path.relative_to(ROOT)}")
else:
    print("Manuscript outline Markdown write skipped for Fig12-only revision.")


In [ ]:
# Final review of whether all outputs assigned to Subagent 12 exist and are non-empty.
# This step only reads file status and gives the notebook run a visible completion summary at the end.
final_status_rows = []
for path in RESPONSIBLE_WRITES[1:]:  # the notebook itself is saved by nbconvert/in-place, so size stability is not checked during the run.
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    final_status_rows.append({
        "file": str(path.relative_to(ROOT)),
        "exists": exists,
        "size_bytes": size,
        "nonzero": bool(exists and size > 0),
    })
final_status_df = pd.DataFrame(final_status_rows)
print(final_status_df.to_string(index=False))
print(f"Final QA status: {qa_status}")
